# Day 20: GPU Generations: Hopper, Ada, Blackwell, Rubin
> *100 Days of Inference* | Layer: **Infrastructure** | Book: *Inference Engineering* Ch 3.2

**Prerequisite:** Day 19

## What problem does this solve?

Inference performance depends on which GPU generation runs the workload. Different generations have different Tensor Core capabilities, memory specifications, and precision support. Understanding the generational trajectory clarifies hardware planning and explains why benchmarks are often hardware-specific.

## Concept Overview

NVIDIA's major GPU architectures for inference:

- **Ampere (A100, 2020):** First with BF16 support, 80GB HBM2e variant; introduced Multi-Instance GPU (MIG).
- **Hopper (H100, 2022):** Added FP8 Tensor Cores, TMA (Tensor Memory Accelerator), 80GB HBM3, Transformer Engine.
- **Ada Lovelace (RTX 4090, 2022):** Consumer/workstation class, 24GB VRAM, limited FP8 support.
- **Blackwell (B200/B100, 2024):** NVFP4 (4-bit float), 192GB HBM3e at 8 TB/s, NVLink 5 at 1.8 TB/s.
- **Rubin (R100, 2026):** 288GB HBM4 at 22 TB/s, 50 PFLOPS FP4, NVLink 6 at 3.6 TB/s per GPU; announced CES 2026, sampling Q4 2026, Vera Rubin NVL72 racks shipping H2 2026.

The most significant jump for inference was **Hopper → FP8**: 2x more GEMM compute per watt on the same operation, plus FP8 E4M3/E5M2 data types that balance range vs precision for weights vs activations. **Blackwell → FP4** is the next one: 2× throughput over FP8 when the model tolerates 4-bit quantization.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"Compute capability: {props.major}.{props.minor}")

PyTorch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA GB10
Compute capability: 12.1


## GPU Generation Comparison
Key specs and inference-relevant features per generation.

In [2]:
gpu_generations = {
    "Ampere (A100 SXM4)": {
        "year": 2020, "arch": "Ampere", "process": "7nm (TSMC)",
        "fp16_tflops": 312, "bf16_tflops": 312, "fp8_tflops": None, "fp4_tflops": None,
        "hbm_gb": 80, "hbm_bw_tb_s": 2.0,
        "nvlink_bw_tb_s": 0.6,
        "key_features": ["3rd gen Tensor Cores", "Multi-Instance GPU (MIG)", "NVLink 3.0"],
        "inference_highlight": "First GPU purpose-built for ML at scale"
    },
    "Hopper (H100 SXM5)": {
        "year": 2022, "arch": "Hopper", "process": "4N (TSMC)",
        "fp16_tflops": 989, "bf16_tflops": 989, "fp8_tflops": 1979, "fp4_tflops": None,
        "hbm_gb": 80, "hbm_bw_tb_s": 3.35,
        "nvlink_bw_tb_s": 0.9,
        "key_features": ["Transformer Engine (FP8)", "TMA (Tensor Memory Accelerator)",
                         "Asynchronous execution", "NVLink 4.0"],
        "inference_highlight": "FP8 Transformer Engine: 2x throughput vs FP16"
    },
    "Ada (RTX 4090)": {
        "year": 2022, "arch": "Ada Lovelace", "process": "4N (TSMC)",
        "fp16_tflops": 165, "bf16_tflops": 165, "fp8_tflops": None, "fp4_tflops": None,
        "hbm_gb": 24, "hbm_bw_tb_s": 1.01,
        "nvlink_bw_tb_s": None,
        "key_features": ["DLSS 3 Frame Generation", "ADA FP8 (limited)", "Shader Execution Reordering"],
        "inference_highlight": "Best consumer GPU for local inference; 24GB VRAM"
    },
    "Blackwell (B200)": {
        "year": 2024, "arch": "Blackwell", "process": "4NP (TSMC)",
        "fp16_tflops": 2250, "bf16_tflops": 2250, "fp8_tflops": 4500, "fp4_tflops": 9000,
        "hbm_gb": 192, "hbm_bw_tb_s": 8.0,
        "nvlink_bw_tb_s": 1.8,
        "key_features": ["5th gen Tensor Cores", "FP4 support", "NVLink 5.0",
                         "2nd gen Transformer Engine", "Confidential Computing"],
        "inference_highlight": "FP4 at 9 PFLOPS: 2x B100, 4.5x H100 (fp8-equivalent)"
    },
    "Rubin (R100)": {
        "year": 2026, "arch": "Rubin", "process": "3nm N3P (TSMC)",
        "fp16_tflops": 8000, "bf16_tflops": 8000, "fp8_tflops": 16000, "fp4_tflops": 50000,
        "hbm_gb": 288, "hbm_bw_tb_s": 22.0,
        "nvlink_bw_tb_s": 3.6,
        "key_features": ["Dual-die 336B transistors", "HBM4", "NVLink 6.0",
                         "3rd gen Transformer Engine", "Vera Arm CPU pairing"],
        "inference_highlight": "FP4 at 50 PFLOPS: 5.6x B200, HBM4 at 22 TB/s (2.75x Blackwell)"
    },
    "DGX Spark (GB10)": {
        "year": 2025, "arch": "Blackwell (desktop)", "process": "4N (TSMC)",
        "fp16_tflops": 1000, "bf16_tflops": 1000, "fp8_tflops": 2000, "fp4_tflops": 4000,
        "hbm_gb": 128, "hbm_bw_tb_s": 4.0,
        "nvlink_bw_tb_s": 0.9,
        "key_features": ["5th gen Tensor Cores (sm_121)", "1 PFLOP FP4", "Arm + Blackwell SoC",
                         "Desk-side 240W form factor"],
        "inference_highlight": "Personal AI supercomputer; models up to 200B params"
    },
}

for name, specs in gpu_generations.items():
    print(f"\n{'='*60}")
    print(f"  {name} ({specs['year']})")
    fp8 = specs['fp8_tflops']; fp4 = specs['fp4_tflops']
    print(f"  FP16: {specs['fp16_tflops']} TFLOPS  |  FP8: {fp8 if fp8 else '—'} TFLOPS  |  FP4: {fp4 if fp4 else '—'} TFLOPS")
    print(f"  HBM: {specs['hbm_gb']}GB @ {specs['hbm_bw_tb_s']} TB/s")
    print(f"  Key: {specs['inference_highlight']}")


  Ampere (A100 SXM4) (2020)
  FP16: 312 TFLOPS  |  FP8: — TFLOPS  |  FP4: — TFLOPS
  HBM: 80GB @ 2.0 TB/s
  Key: First GPU purpose-built for ML at scale

  Hopper (H100 SXM5) (2022)
  FP16: 989 TFLOPS  |  FP8: 1979 TFLOPS  |  FP4: — TFLOPS
  HBM: 80GB @ 3.35 TB/s
  Key: FP8 Transformer Engine: 2x throughput vs FP16

  Ada (RTX 4090) (2022)
  FP16: 165 TFLOPS  |  FP8: — TFLOPS  |  FP4: — TFLOPS
  HBM: 24GB @ 1.01 TB/s
  Key: Best consumer GPU for local inference; 24GB VRAM

  Blackwell (B200) (2024)
  FP16: 2250 TFLOPS  |  FP8: 4500 TFLOPS  |  FP4: 9000 TFLOPS
  HBM: 192GB @ 8.0 TB/s
  Key: FP4 at 9 PFLOPS: 2x B100, 4.5x H100 (fp8-equivalent)

  Rubin (R100) (2026)
  FP16: 8000 TFLOPS  |  FP8: 16000 TFLOPS  |  FP4: 50000 TFLOPS
  HBM: 288GB @ 22.0 TB/s
  Key: FP4 at 50 PFLOPS: 5.6x B200, HBM4 at 22 TB/s (2.75x Blackwell)

  DGX Spark (GB10) (2025)
  FP16: 1000 TFLOPS  |  FP8: 2000 TFLOPS  |  FP4: 4000 TFLOPS
  HBM: 128GB @ 4.0 TB/s
  Key: Personal AI supercomputer; models up to 200B pa

## Transformer Engine: H100's Inference Killer Feature
The Transformer Engine dynamically selects FP8 or FP16 precision per layer, automatically managing the scale factors.

In [5]:
# Simulate Transformer Engine's per-layer precision selection
import torch

def transformer_engine_forward(layer_stats: dict) -> str:
    # TE measures activation range and chooses FP8 when it's safe
    amax = layer_stats.get("activation_amax", 1.0)
    scale_factor = 448.0 / amax  # FP8 E4M3 max = 448

    # If scale brings activations into FP8 range without overflow: use FP8
    if amax < 400 and scale_factor > 0.1:
        return "FP8 E4M3"
    elif amax < 65504:  # FP16 max
        return "FP16"
    else:
        return "BF16"

# Simulate a 32-layer transformer forward pass
layers = [
    {"name": f"Layer {i+1}", "activation_amax": abs(np.random.normal(50, 200))}
    for i in range(32)
]
# Inject outlier layers
layers[5]["activation_amax"] = 500
layers[15]["activation_amax"] = 600
layers[20]["activation_amax"] = 70000  # extreme outlier -> BF16

fp8_count = 0; fp16_count = 0; bf16_count = 0
for layer in layers:
    dtype = transformer_engine_forward(layer)
    if dtype == "FP8 E4M3": fp8_count += 1
    elif dtype == "FP16": fp16_count += 1
    else: bf16_count += 1

total = len(layers)
print(f"Transformer Engine precision selection ({total} layers):")
print(f"  FP8 E4M3: {fp8_count}/{total} ({fp8_count/total:.0%}) layers -> 2x compute")
print(f"  FP16:     {fp16_count}/{total} ({fp16_count/total:.0%}) layers -> 1x compute")
print(f"  BF16:     {bf16_count}/{total} ({bf16_count/total:.0%}) layers -> 1x compute")
print(f"  Effective compute gain: {(fp8_count*2 + fp16_count + bf16_count)/total:.2f}x average")

Transformer Engine precision selection (32 layers):
  FP8 E4M3: 28/32 (88%) layers -> 2x compute
  FP16:     3/32 (9%) layers -> 1x compute
  BF16:     1/32 (3%) layers -> 1x compute
  Effective compute gain: 1.88x average


## FP4 Tensor Cores: Blackwell's Leap
FP4 (4-bit floating point) doubles FP8 throughput. With careful quantization, quality is maintained for inference.

In [6]:
# FP4 E2M1 format analysis
# 4 bits: 1 sign + 2 exponent + 1 mantissa

def fp4_analysis():
    # FP4 E2M1 representable values
    values = set()
    for sign in [0, 1]:
        for exp in range(4):  # 2^2 = 4 exponent values
            for mant in range(2):  # 2^1 = 2 mantissa values
                if exp == 0:  # subnormal
                    val = (-1)**sign * mant * 2**(-1)
                else:
                    val = (-1)**sign * (1 + mant * 0.5) * 2**(exp - 1)
                values.add(round(val, 4))
    return sorted(values)

fp4_vals = fp4_analysis()
print(f"FP4 E2M1 representable values ({len(fp4_vals)}):")
print(fp4_vals)
print(f"Max positive: {max(fp4_vals)}")
print(f"Min positive (nonzero): {min(v for v in fp4_vals if v > 0):.4f}")

# Compare format ranges
formats_range = {
    "FP32":    (2**127 * (2 - 2**-23), 2**-149),
    "BF16":    (2**127 * (2 - 2**-7), 2**-133),
    "FP16":    (65504, 5.96e-8),
    "FP8 E4M3": (448, 2**-9),
    "FP8 E5M2": (57344, 2**-16),
    "FP4 E2M1": (6.0, 0.5),
}

print(f"\n{'Format':12s} {'Max':>15} {'Min positive':>15} {'Dynamic range':>15}")
for name, (max_v, min_v) in formats_range.items():
    dr = max_v / min_v
    print(f"{name:12s} {max_v:>15.2e} {min_v:>15.2e} {dr:>15.2e}")

FP4 E2M1 representable values (15):
[-6.0, -4.0, -3.0, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]
Max positive: 6.0
Min positive (nonzero): 0.5000

Format                   Max    Min positive   Dynamic range
FP32                3.40e+38        1.40e-45        2.43e+83
BF16                3.39e+38        9.18e-41        3.69e+78
FP16                6.55e+04        5.96e-08        1.10e+12
FP8 E4M3            4.48e+02        1.95e-03        2.29e+05
FP8 E5M2            5.73e+04        1.53e-05        3.76e+09
FP4 E2M1            6.00e+00        5.00e-01        1.20e+01


## Throughput Projection: H100 → B200 → Rubin
Model the inference throughput trajectory for decode-bound workloads.

In [7]:
# Decode throughput model: bound by HBM bandwidth
def decode_throughput_tps(model_params_b, hbm_bw_tb_s, quantization_bytes=2):
    weight_bytes = model_params_b * 1e9 * quantization_bytes
    tps = hbm_bw_tb_s * 1e12 / weight_bytes
    return tps

models = [("Llama-3-8B", 8), ("Llama-3-70B", 70), ("Llama-3-405B", 405)]
gpus_timeline = [
    ("A100 SXM4 (2020)",   2.0),
    ("H100 SXM5 (2022)",   3.35),
    ("GH200 (2023)",       4.0),
    ("B200 (2024)",        8.0),
    ("R100 Rubin (2026)", 22.0),  # confirmed HBM4 bandwidth at GTC/CES 2026
]

print("Single-GPU decode throughput (tokens/sec, FP16, batch=1):")
print(f"{'':30s}", end="")
for name, _ in models:
    print(f" {name:>15}", end="")
print()

for gpu_name, bw in gpus_timeline:
    print(f"{gpu_name:30s}", end="")
    for _, params in models:
        tps = decode_throughput_tps(params, bw, quantization_bytes=2)
        print(f" {tps:>15.0f}", end="")
    print()

print("\nNote: decode is memory-bound — throughput tracks HBM bandwidth almost exactly.")
print("Rubin's 22 TB/s HBM4 is ~2.75x Blackwell's 8 TB/s, translating directly to ~2.75x decode tokens/sec at FP16.")

Single-GPU decode throughput (tokens/sec, FP16, batch=1):
                                    Llama-3-8B     Llama-3-70B    Llama-3-405B
A100 SXM4 (2020)                           125              14               2
H100 SXM5 (2022)                           209              24               4
GH200 (2023)                               250              29               5
B200 (2024)                                500              57              10
R100 Rubin (2026)                         1375             157              27

Note: decode is memory-bound — throughput tracks HBM bandwidth almost exactly.
Rubin's 22 TB/s HBM4 is ~2.75x Blackwell's 8 TB/s, translating directly to ~2.75x decode tokens/sec at FP16.


## Experiments: Try These

1. Check the local GPU's compute capability (`torch.cuda.get_device_properties(0)`) and look up what Tensor Core operations it supports — can it run FP8? FP4?
2. Run a matmul benchmark in FP16 vs BF16 — measure whether the GPU achieves the same throughput for both formats.
3. Compute: at what model size does B200's FP4 capability (9 PFLOPS) become more throughput-efficient than H100 FP8 (2 PFLOPS) for a given tokens/sec target? How does Rubin's 50 PFLOPS FP4 change the answer?

## Key Takeaways

- Each GPU generation roughly doubles inference throughput: A100→H100 (2x via FP8), H100→B200 (2x FP8, 4x FP4), B200→R100 (2.5–5x via HBM4 + 50 PFLOPS FP4).
- The Transformer Engine's automatic mixed-precision selection is the key H100 inference feature — most FP8 gains land without manual quantization effort.
- FP4 (Blackwell) is the next frontier: 9 PFLOPS on B200, 50 PFLOPS on Rubin, enabling frontier-model serving at unprecedented throughput.
- Decode is memory-bound — the HBM bandwidth number on the spec sheet is a very good first-order tokens/sec predictor.
- **What's next:** Day 21 — Multi-GPU Instances and Multi-Instance GPU (MIG).

## References
- *Inference Engineering* Ch 3.2 — Philip Kiely
- [NVIDIA H100 Tensor Core GPU Architecture whitepaper](https://resources.nvidia.com/en-us-tensor-core)
- [NVIDIA Vera Rubin NVL72 product page](https://www.nvidia.com/en-us/data-center/vera-rubin-nvl72/)
- [Hopper FP8 paper (Micikevicius et al., 2022)](https://arxiv.org/abs/2209.05433)